# Calories Burnt Prediction

Predict calories burned during exercise using an **XGBoost regression model**.

This notebook covers:
1. Environment setup and imports
2. Loading and combining the exercise and calories datasets
3. Data inspection and cleaning
4. Exploratory data analysis and visualization
5. Feature/target preparation
6. Train/test split
7. XGBoost model training
8. Model evaluation using Mean Absolute Error (MAE)
9. Saving the trained model and feature names

## 1. Imports

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")
RANDOM_STATE = 2

## 2. Load the Data

Place the original `calories.csv` and `exercise.csv` files in `data/` before running this notebook.

In [ ]:
DATA_DIR = "../data"

calories_path = os.path.join(DATA_DIR, "calories.csv")
exercise_path = os.path.join(DATA_DIR, "exercise.csv")

if not os.path.exists(calories_path) or not os.path.exists(exercise_path):
    raise FileNotFoundError(
        "Place calories.csv and exercise.csv inside the project's data/ directory."
    )

calories = pd.read_csv(calories_path)
exercise_data = pd.read_csv(exercise_path)

print("Calories dataset:", calories.shape)
print("Exercise dataset:", exercise_data.shape)

In [ ]:
display(calories.head())
display(exercise_data.head())

## 3. Combine and Inspect the Data

In [ ]:
calories_data = pd.concat(
    [exercise_data.reset_index(drop=True), calories["Calories"].reset_index(drop=True)],
    axis=1
)

print("Combined shape:", calories_data.shape)
calories_data.head()

In [ ]:
calories_data.info()

In [ ]:
missing_values = calories_data.isnull().sum()
print(missing_values)

In [ ]:
calories_data.describe()

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=calories_data, x="Gender")
plt.title("Gender Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(calories_data["Age"], kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")

sns.histplot(calories_data["Height"], kde=True, ax=axes[1])
axes[1].set_title("Height Distribution")

sns.histplot(calories_data["Weight"], kde=True, ax=axes[2])
axes[2].set_title("Weight Distribution")

plt.tight_layout()
plt.show()

## 5. Encode Categorical Data and Analyze Correlations

In [ ]:
calories_data["Gender"] = calories_data["Gender"].map({"male": 0, "female": 1})

if calories_data["Gender"].isnull().any():
    raise ValueError("Unexpected values were found in the Gender column.")

calories_data.head()

In [ ]:
correlation = calories_data.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation,
    cbar=True,
    square=True,
    fmt=".1f",
    annot=True,
    annot_kws={"size": 8},
    cmap="Blues"
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 6. Prepare Features and Target

In [ ]:
X = calories_data.drop(columns=["User_ID", "Calories"], errors="ignore")
y = calories_data["Calories"]

print("Features:", X.columns.tolist())
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 7. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 8. Train the XGBoost Regressor

In [ ]:
model = XGBRegressor(
    random_state=RANDOM_STATE,
    objective="reg:squarederror"
)

model.fit(X_train, y_train)

## 9. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

display(comparison.head(10))

## 10. Save the Model and Feature Names

The trained artifacts are saved in `models/` so they can be reused by an application without retraining.

In [ ]:
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "xgb_model.pkl")
features_path = os.path.join(MODEL_DIR, "feature_cols.pkl")

joblib.dump(model, model_path)
joblib.dump(X.columns.tolist(), features_path)

print(f"Model saved to: {model_path}")
print(f"Feature names saved to: {features_path}")

## 11. Reproducible Prediction Example

The model expects the same seven input features used during training.

In [ ]:
sample_input = pd.DataFrame([{
    "Gender": 0,
    "Age": 25,
    "Height": 170,
    "Weight": 70,
    "Duration": 30,
    "Heart_Rate": 110,
    "Body_Temp": 37
}])

sample_prediction = model.predict(sample_input)[0]
print(f"Estimated calories burned: {sample_prediction:.2f} kcal")

## Conclusion

The notebook trains an XGBoost regression model to estimate calories burned from exercise-related attributes. The MAE printed above provides the model's average absolute prediction error on the held-out test set.

For deployment, keep the trained model in `models/` and use the same feature order and preprocessing shown in this notebook.